# Sabeel: pothole detector training

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

Choices, aimed at high test metrics with fast CPU inference on our API:
- **YOLO11s**: clearly more accurate than YOLOv8n, still around 100 ms on a CPU at 640 px. If the latency in cell 8 is too slow for us, set `MODEL = "yolo11n.pt"` and retrain.
- **Two data sources**: RDD2022 potholes (dashcam views from 6 countries) plus 300 close-up pothole photos from Hugging Face (Roboflow, CC BY 4.0; like a citizen's phone snap). Pothole-free road photos are added so the model learns what a normal road looks like.
- **Held-out test split** (70/15/15). The confidence threshold is tuned on **val**, and the final numbers come from **test**.

The dataset is built on Colab's local disk; weights, plots and metrics go to Google Drive (`MyDrive/sabeel`), so a disconnect doesn't lose them.
Start with a quick test (`EPOCHS = 1`), then set `EPOCHS = 100` (roughly 3–5 h on a T4; keep the tab open). If Colab disconnects, see **Resume** at the bottom.

In [ ]:
# 1. Settings, GPU check, Drive
MODEL = "yolo11s.pt"  # "yolo11n.pt" = faster, a bit less accurate
EPOCHS = 100          # use 1 for a quick pipeline test
IMGSZ = 640           # keep 640: matches ml-training/inference.py and keeps CPU latency low
NEG_RATIO = 0.15      # pothole-free road photos, as a share of pothole photos
RDD_URL = "https://ndownloader.figshare.com/files/38030910"   # official RDD2022 on figshare (13.3 GB); the old S3 link returns 403

import subprocess
assert subprocess.run(["nvidia-smi"]).returncode == 0, "No GPU: Runtime → Change runtime type → T4 GPU"

from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/sabeel"
!mkdir -p {SAVE}

In [ ]:
# 2. Install
!pip install -q -U ultralytics onnxruntime

In [ ]:
# 3. Get the data. Fast path: restore the built dataset cached on Drive by an earlier session (~1 min).
#    Slow path (first time only): download and unpack RDD2022 (13.3 GB from figshare, 15-40 min).
import os, glob
CACHE = f"{SAVE}/data_cache.zip"
CACHED = os.path.exists(CACHE)
if CACHED:
    !unzip -q -o "{CACHE}" -d /content
    print("restored dataset from Drive:", {s: len(glob.glob(f'/content/data/{s}/images/*')) for s in ('train', 'val', 'test')})
else:
    import os, glob
    def n_xml(): return len(glob.glob("/content/rdd/**/*.xml", recursive=True))
    if n_xml() == 0:                       # re-downloads if an earlier attempt failed or was cut short
        !rm -rf /content/rdd /content/rdd.zip
        !wget --show-progress -O /content/rdd.zip "{RDD_URL}"
        !ls -la /content/rdd.zip && file /content/rdd.zip
        !mkdir -p /content/rdd && unzip -q /content/rdd.zip -d /content/rdd && rm /content/rdd.zip
        # the archive contains one zip per country (maybe nested), unpack until none are left
        while zips := glob.glob("/content/rdd/**/*.zip", recursive=True):
            for z in zips:
                subprocess.run(["unzip", "-q", "-o", z, "-d", os.path.dirname(z)], check=True)
                os.remove(z)
    !df -h /content | tail -1
    !find /content/rdd -maxdepth 5 -type d | head -40
    print("XML annotation files:", n_xml(), "| images:", len(glob.glob("/content/rdd/**/*.jpg", recursive=True)))
    assert n_xml(), "No annotations: the download failed (see the wget / file output above)"

In [ ]:
# 4. Build one dataset: RDD2022 D40 + close-up pothole photos + negatives, split 70/15/15.
#    Skipped when cell 3 restored it from Drive; after a fresh build it is cached to Drive for next time.
if CACHED:
    print("dataset restored from Drive, nothing to build")
else:
    import random, shutil, xml.etree.ElementTree as ET
    from pathlib import Path
    from PIL import Image

    OUT = Path("/content/data")
    shutil.rmtree(OUT, ignore_errors=True)
    random.seed(42)
    pos, neg = [], []   # (image_path, yolo_label_lines)

    # 4a. RDD2022: Pascal VOC XML, keep only D40 (pothole) boxes
    n_rdd = 0
    # match each XML to its image by filename, wherever it sits in the tree
    images = {p.stem: p for ext in ("jpg", "jpeg", "png") for p in Path("/content/rdd").rglob(f"*.{ext}")}
    xmls = list(Path("/content/rdd").rglob("*.xml"))
    print("XML files:", len(xmls), "| images:", len(images))
    for x in xmls:
        img = images.get(x.stem)
        if img is None:
            continue
        r = ET.parse(x).getroot()
        W, H = int(float(r.findtext("size/width") or 0)), int(float(r.findtext("size/height") or 0))
        if not W or not H:
            W, H = Image.open(img).size
        lines = []
        for o in r.iter("object"):
            if o.findtext("name", "").strip() != "D40":
                continue
            b = o.find("bndbox")
            x1, y1, x2, y2 = (float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
            if x2 > x1 and y2 > y1:
                lines.append(f"0 {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
        (pos if lines else neg).append((img, lines))
        n_rdd += bool(lines)
    print("RDD2022 pothole images:", n_rdd, "| pothole-free:", len(neg))
    assert n_rdd, "0 RDD2022 pothole images: the folder layout differs, check the tree printed by cell 3"

    # 4b. Close-up pothole photos (Roboflow "Potholes Detection", CC BY 4.0, 300 images, YOLO format) from Hugging Face.
    #     Optional: skipped if the download fails.
    n_ul = 0
    try:
        from huggingface_hub import snapshot_download
        d = Path(snapshot_download("Ryukijano/Pothole-detection-Yolov8", repo_type="dataset", local_dir="/content/closeup"))
        for split in ("train", "valid", "test"):
            for img in (d/split/"images").glob("*.jpg"):
                lbl = d/split/"labels"/f"{img.stem}.txt"
                lines = []
                if lbl.exists():
                    for row in lbl.read_text().split("\n"):
                        v = [float(t) for t in row.split()[1:]]
                        if len(v) == 4:                                  # box: cx cy w h
                            cx, cy, w, h = v
                            if w > 0 and h > 0:
                                lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                        elif len(v) >= 6:                                # polygon -> box
                            xs, ys = v[0::2], v[1::2]
                            x1, x2, y1, y2 = max(0, min(xs)), min(1, max(xs)), max(0, min(ys)), min(1, max(ys))
                            if x2 > x1 and y2 > y1:
                                lines.append(f"0 {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}")
                if lines:
                    pos.append((img, lines))
                    n_ul += 1
    except Exception as e:
        print("close-up pothole set skipped:", e)
    print("close-up pothole images:", n_ul)

    # 4c. Mix, split, write
    random.shuffle(neg)
    items = pos + neg[: int(len(pos) * NEG_RATIO)]
    random.shuffle(items)
    n = len(items)
    splits = {"train": items[: int(.7*n)], "val": items[int(.7*n): int(.85*n)], "test": items[int(.85*n):]}
    for split, rows in splits.items():
        (OUT/split/"images").mkdir(parents=True, exist_ok=True)
        (OUT/split/"labels").mkdir(parents=True, exist_ok=True)
        for i, (img, lines) in enumerate(rows):
            name = f"{i:06d}_{img.stem}"                     # unique even if two sources share a filename
            shutil.copy(img, OUT/split/"images"/f"{name}{img.suffix}")
            (OUT/split/"labels"/f"{name}.txt").write_text("\n".join(lines))

    Path("/content/data.yaml").write_text(
        "path: /content/data\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n  0: pothole\n")
    print("total pothole images:", len(pos), "|", {k: len(v) for k, v in splits.items()})

    !cd /content && zip -q -r -0 "{CACHE}" data data.yaml
    print("cached dataset to Drive:", CACHE)

In [ ]:
# 5. Train (weights saved to Drive after every epoch)
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data="/content/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=-1, device=0,
            patience=25, cos_lr=True, close_mosaic=10, workers=2,
            project=f"{SAVE}/runs", name="pothole", exist_ok=True)

In [ ]:
# 6. Tune the confidence threshold on VAL (max F1), then score once on TEST
import json, numpy as np
best = YOLO(f"{SAVE}/runs/pothole/weights/best.pt")

v = best.val(data="/content/data.yaml", split="val", imgsz=IMGSZ, plots=False,
             project=f"{SAVE}/runs", name="val", exist_ok=True)
try:
    f1 = np.asarray(v.box.f1_curve).mean(0)
    BEST_CONF = round(float(np.asarray(v.box.px)[f1.argmax()]), 3)
    print("best confidence threshold on val:", BEST_CONF, "| F1:", round(float(f1.max()), 3))
except Exception as e:
    BEST_CONF = 0.25
    print("threshold tuning unavailable in this ultralytics version, using 0.25:", e)

m = best.val(data="/content/data.yaml", split="test", imgsz=IMGSZ,
             project=f"{SAVE}/runs", name="test", exist_ok=True)
try:
    tf1 = np.asarray(m.box.f1_curve).mean(0)
    test_f1 = round(float(tf1[int(np.abs(np.asarray(m.box.px) - BEST_CONF).argmin())]), 4)
except Exception:
    test_f1 = None
metrics = {
    "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ,
    "test_images": len(glob.glob("/content/data/test/images/*")),
    "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
    "mAP50": round(float(m.box.map50), 4), "mAP50-95": round(float(m.box.map), 4),
    "conf_threshold": BEST_CONF, "test_F1_at_threshold": test_f1,
}
print(metrics)

In [ ]:
# 7. Export to ONNX (fixed 640x640, same contract as ml-training/inference.py)
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
print("ONNX:", onnx_path)

In [ ]:
# 8. CPU latency of the ONNX model (Colab has 2 slow vCPUs, so our server will be at least this fast)
import time, onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
x = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float32)
for _ in range(3): sess.run(None, {inp.name: x})
t = []
for _ in range(20):
    t0 = time.perf_counter(); sess.run(None, {inp.name: x}); t.append((time.perf_counter() - t0) * 1000)
metrics["cpu_ms_colab"] = round(float(np.median(t)), 1)
metrics["onnx_output_shape"] = [int(s) for s in sess.run(None, {inp.name: x})[0].shape]   # expect [1, 5, 8400]
print(f"median CPU inference: {metrics['cpu_ms_colab']} ms | output {metrics['onnx_output_shape']}")
Path(f"{SAVE}/test_metrics.json").write_text(json.dumps(metrics, indent=2))
!ls -la {SAVE}/runs/pothole/weights {SAVE}

In [ ]:
# 9. Visual check: predictions on a few test images at the tuned threshold
from IPython.display import Image as Show, display
for p in sorted(glob.glob("/content/data/test/images/*.jpg"))[:8]:
    best.predict(p, conf=BEST_CONF, imgsz=IMGSZ, verbose=False)[0].save(filename="/content/pred.jpg")
    display(Show("/content/pred.jpg", width=480))

## Resume after a disconnect
**Only if a training run was cut off before it finished.** A finished run (like the 1-epoch test) can't be resumed: to train longer, change `EPOCHS` in cell 1 and run cell 5 again.

Run cells 1–4 (they restore the dataset from Drive in about a minute), then the cell below, then cells 6–9.

In [ ]:
import torch
from ultralytics import YOLO
LAST = f"{SAVE}/runs/pothole/weights/last.pt"
ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
# a finished run has its optimizer stripped and epoch set to -1; resuming it makes ultralytics start an unrelated default training
if ckpt.get("optimizer") is None or ckpt.get("epoch", -1) < 0:
    print("That run already finished, so there is nothing to resume. Set EPOCHS in cell 1, rerun cell 1, then run cell 5.")
else:
    print(f"resuming from epoch {ckpt['epoch'] + 1}")
    YOLO(LAST).train(resume=True)

## Then
From Google Drive → `MyDrive/sabeel`, download:
- `runs/pothole/weights/best.onnx` (the new model) and `best.pt` (for retraining)
- `test_metrics.json` (the numbers for the README and the judges)
- `runs/pothole/results.png`, `runs/test/confusion_matrix.png`, `runs/test/PR_curve.png` (for the slides)